# gpt-oss-20b — MXFP4-on-T4 probe (go/no-go, no upload)

Loads gpt-oss-20b (native MXFP4) on a Kaggle T4 and captures ONE document. Success = model loads on
sm_75, `.mlp.router` x24 discovered, and the SelectionMismatch gate passes. **Before running:** GPU
**T4 x2**, **Internet On**. Run top-to-bottom; do not restart the kernel. Paste back the final
`PROBE OK` / failure line.


In [ ]:
# Cell 1 — runtime audit.
import subprocess
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,compute_cap",
     "--format=csv"], capture_output=True, text=True).stdout, flush=True)
print("EXPECT two rows, compute_cap 7.5.")


In [ ]:
# Cell 2 — install vLLM + the MXFP4 kernel deps (triton>=3.4, kernels). No restart after.
VLLM_VERSION = "0.10.2"
import subprocess, sys


def sh(args):
    print("$", " ".join(args), flush=True)
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "")[-3000:], flush=True)
    print("exit:", p.returncode, flush=True)
    return p.returncode


sh([sys.executable, "-m", "pip", "uninstall", "-y",
    "vllm", "torch", "torchvision", "torchaudio", "transformers"])
sh([sys.executable, "-m", "pip", "install", "-q",
    f"vllm=={VLLM_VERSION}", "transformers==4.55.2", "huggingface_hub>=0.34.0,<1.0"])
# MXFP4 needs a recent triton + the kernels package; without them transformers dequantizes to bf16
# (~40 GB) and OOMs a T4. Constrain huggingface_hub<1.0 IN THE SAME resolve (no -U): kernels'
# latest pulls hub 2.0.0, which transformers 4.55.2 rejects (needs hub>=0.34,<1.0).
sh([sys.executable, "-m", "pip", "install", "-q", "triton>=3.4", "kernels",
    "huggingface_hub>=0.34.0,<1.0"])
# Belt-and-suspenders: force hub back into range in case anything above bumped it past 1.0.
sh([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.34.0,<1.0"])
sh([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"])

chk = subprocess.run(
    [sys.executable, "-c",
     "import torch, vllm, triton, huggingface_hub as h; "
     "print('IMPORT_OK vllm', vllm.__version__, 'triton', triton.__version__, 'hub', h.__version__)"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(chk.stdout, flush=True)
print(">>> Proceed only if IMPORT_OK printed. DO NOT restart the kernel.")


In [ ]:
# Cell 3 — clone the repo, put src/ on sys.path AND PYTHONPATH.
import os, subprocess, sys
from pathlib import Path

GIT_URL = "https://github.com/ryzewtf/GenAI-IA-1.git"
GIT_REF = "VLLM_PORT"
REPO = Path("/kaggle/working/repo")


def run(cmd, cwd=None, check=True, quiet=False):
    if not quiet:
        print("$", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=cwd and str(cwd), text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.stdout and not quiet:
        print(p.stdout, flush=True)
    if check and p.returncode != 0:
        raise SystemExit(f"FAILED ({p.returncode}): {' '.join(str(c) for c in cmd)}")
    return p


url = GIT_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if tok:
        url = GIT_URL.replace("https://", f"https://{tok}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning anonymously")

if REPO.exists():
    run(["git", "fetch", "--all", "--tags"], cwd=REPO)
    run(["git", "checkout", GIT_REF], cwd=REPO)
    run(["git", "pull", "--ff-only"], cwd=REPO, check=False)
else:
    run(["git", "clone", url, str(REPO)])
    run(["git", "checkout", GIT_REF], cwd=REPO)
print("repo at", run(["git", "rev-parse", "HEAD"], cwd=REPO, quiet=True).stdout.strip())

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(REPO)


In [ ]:
# Cell 4 — load gpt-oss once, discover .mlp.router, capture ONE doc, fire the gate.
import numpy as np
from src.runtime.runner import load_model_meta
from src.capture.vllm_collect import VLLMCaptureEngine, spec_and_gating_for
from src.capture.vllm_trace import DocumentTrace, discover_router_and_experts

meta = load_model_meta("configs/models.yaml", "gpt-oss-20b")
vllm = meta["vllm"]
spec, gating = spec_and_gating_for(meta)
print("spec:", (spec.n_moe_layers, spec.n_experts, spec.top_k, spec.hidden_dim),
      "| softmax", gating.softmax, "has_router_bias", gating.has_router_bias)

engine = VLLMCaptureEngine(
    model_id=vllm["model_id"], tensor_parallel_size=1, spec=spec, gating=gating,
    router_suffix=vllm["router_suffix"], experts_suffix=vllm["experts_suffix"],
    max_model_len=2048, gpu_memory_utilization=float(vllm.get("gpu_memory_utilization", 0.90)),
    dtype="float16", seed=0,
)
engine.load()  # <-- the MXFP4-on-T4 moment: fails here if sm_75 cannot load the model
try:
    model = engine.llm.llm_engine.model_executor.driver_worker.model_runner.model
    _, _, gnames, enames = discover_router_and_experts(
        model, router_suffix=vllm["router_suffix"], experts_suffix=vllm["experts_suffix"],
        n_expected=spec.n_moe_layers)
    print(f"discovered {len(gnames)} routers, e.g. {gnames[0]} .. {gnames[-1]}")

    ids = engine.tokenize("The mixture-of-experts router selects a few experts per token.")
    captured = engine.capture_document(ids)
    n = len(ids)
    trace = DocumentTrace(spec, n_tokens=n, capture_mask=[False] * n, gating=gating)
    for L in range(spec.n_moe_layers):
        logits, topk, router_input = captured[L]
        trace.put_layer(L, logits=logits, vllm_topk=topk, router_input=router_input)  # gate fires here
    print(f"SelectionMismatch gate PASSED over {n} tokens x {spec.n_moe_layers} layers")
    print("PROBE OK — gpt-oss loads and captures faithfully on T4; safe to run the campaign notebook")
finally:
    engine.remove()
